In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):          # x: [B, L, C]
        front = x[:, 0:1 , :].repeat(1, (self.kernel_size-1)//2, 1 )
        end = x[:, -1:, :].repeat(1, (self.kernel_size-1)//2, 1 )
        x = torch.cat([front,x,end],dim = 1)
        x = self.avg(x.permute(0,2,1))
        x = x.permute(0,2,1)
        return x

In [7]:
class series_decomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = moving_avg(kernel_size,1) # BLC


    def forward(self, x):          # x: [B, L, C]
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend

In [14]:
class Model(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.series_decomp = series_decomp(kernel_size)
        self.Linear_Seasonal = nn.Linear(seq_len,pred_len)
        self.Linear_Trend = nn.Linear(seq_len,pred_len) 

    def forward(self, x):          # x: [B, L, C]
        seasonal, trend = self.series_decomp(x)
        seasonal = seasonal.permute(0,2,1)
        trend = trend.permute(0,2,1)
        seasonal = self.Linear_Seasonal(seasonal)
        trend = self.Linear_Trend(trend)
        x = seasonal + trend
        x = x.permute(0,2,1)
        return x

In [2]:
path = '../../data/ETTh1.csv'

df = pd.read_csv(path)

df.head(5)

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


In [15]:
model = Model(seq_len=336, pred_len=96)
print(model(torch.randn(32, 336, 7)).shape)   # kỳ vọng: [32, 96, 7]

torch.Size([32, 96, 7])


In [18]:
print(df.shape)
print(df.columns)

(17420, 8)
Index(['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT'], dtype='object')


In [19]:
data = df.iloc[:, 1:].values 
print(data.shape)               

(17420, 7)


In [26]:
from torch.utils.data import Dataset

class ETTDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data = torch.tensor(data, dtype=torch.float32)   # [N, 7]
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - (self.seq_len+self.pred_len) + 1
        # TODO 1: số cặp (x, y) — viết theo seq_len, pred_len, len(self.data)
        

    def __getitem__(self, i):
        x = self.data[i:i+ self.seq_len]
        y = self.data[i+ self.seq_len:i+self.seq_len+self.pred_len]
        return x,y


ds = ETTDataset(train_data, 336, 96)
print(len(ds))            # kỳ vọng: 8209
x, y = ds[0]
print(x.shape, y.shape)   # kỳ vọng: [336, 7] [96, 7]

8209
torch.Size([336, 7]) torch.Size([96, 7])


In [27]:
scaler = StandardScaler()
scaler.fit(data[:train_end])
data_scaled = scaler.transform(data)

train_end = 12 * 30 * 24       
seq_len, pred_len = 336, 96

train_data = data_scaled[:train_end]
val_data = data_scaled[train_end - seq_len : val_end]
test_data = data_scaled[val_end - seq_len : test_end]

train_ds = ETTDataset(train_data, seq_len, pred_len)
val_ds   = ETTDataset(val_data,   seq_len, pred_len)
test_ds  = ETTDataset(test_data,  seq_len, pred_len)
print(len(train_ds), len(val_ds), len(test_ds))

8209 2785 2785


In [ ]:
from torch.utils.data import DataLoader
torch.manual_seed(42)


train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
criterion = nn.MSELoss()
model = Model(336, 96)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)


print(train_loss)

0.4028541429497391


In [37]:
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x,y in loader:
            pred = model(x)
            loss = criterion(pred,y)
            total_loss += loss.item() / len(loader)
    model.train()
    return total_loss   

In [38]:
evaluate(model, val_loader, criterion)

0.7164908559484918

In [41]:
def train_one_epoch(model, loader, criterion, optimizer):
    train_loss = 0
    for x, y in loader:      # x: [32, 336, 7], y: [32, 96, 7]
        pred = model(x)
        loss = criterion(pred,y)
        train_loss += loss.item() / len(loader)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return train_loss


torch.manual_seed(42)
model = Model(336, 96)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)
best_val = float("inf")
patience, counter = 3, 0

for epoch in range(10):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss   = evaluate(model, val_loader, criterion)
    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]
    print(f"epoch {epoch+1}: train {train_loss:.4f} | val {val_loss:.4f} | lr {lr_now:.6f}")
    if(val_loss < best_val):
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "best.pt")
    else:
        counter += 1
        if(counter >= patience):
            break

model.load_state_dict(torch.load("best.pt"))
    # TODO B: logic early stopping
    #   - tốt hơn best_val → cập nhật best_val, lưu trọng số, reset counter
    #   - không tốt hơn   → counter += 1, nếu counter >= patience thì break

epoch 1: train 0.4029 | val 0.7165 | lr 0.002500
epoch 2: train 0.3555 | val 0.6797 | lr 0.001250
epoch 3: train 0.3428 | val 0.6751 | lr 0.000625
epoch 4: train 0.3372 | val 0.6431 | lr 0.000313
epoch 5: train 0.3341 | val 0.6442 | lr 0.000156
epoch 6: train 0.3324 | val 0.6491 | lr 0.000078
epoch 7: train 0.3312 | val 0.6581 | lr 0.000039


<All keys matched successfully>

In [ ]:
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

test_mse = evaluate(model, test_loader, nn.MSELoss())
test_mae = evaluate(model, test_loader, nn.L1Loss())
print(f"Test MSE: {test_mse:.4f} | MAE: {test_mae:.4f}")

Test MSE: 0.3779 | MAE: 0.3999


: 